In [1]:
# ============ SETUP AND IMPORTS ============
import os, sys, json, time, gc, math, copy, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Set, Any
import warnings
warnings.filterwarnings('ignore')

# Scientific computing
from scipy.ndimage import gaussian_filter, maximum_filter, label, find_objects
from scipy.optimize import linear_sum_assignment
from scipy.spatial import KDTree, cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import norm

# Machine Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Image processing
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

PyTorch version: 2.10.0+cpu
CUDA available: False


In [2]:
# Physical voxel scale (z, y, x) in micrometres per voxel
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

@dataclass
class ImageVolume:
    path: str
    shape: tuple
    dtype: np.dtype
    chunk: tuple
    
    @property
    def n_t(self) -> int:
        return int(self.shape[0])
    
    def frame(self, t: int) -> np.ndarray:
        return _read_chunk(self.path, t, self.shape, self.dtype)

def open_image(zarr_path: str) -> ImageVolume:
    with open(os.path.join(zarr_path, "0", "zarr.json")) as f:
        meta = json.load(f)
    shape = tuple(int(s) for s in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    return ImageVolume(path=zarr_path, shape=shape, dtype=dtype, chunk=None)

def _read_chunk(zarr_path: str, t: int, shape: tuple, dtype: np.dtype) -> np.ndarray:
    """Read and decode one timepoint chunk -> (Z, Y, X)."""
    frame_shape = shape[1:]
    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    
    try:
        import blosc2
        with open(chunk_path, "rb") as f:
            raw = f.read()
        dec = blosc2.decompress(raw)
        arr = np.frombuffer(dec, dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            return arr.reshape(frame_shape).copy()
    except:
        import zarr
        z = zarr.open(os.path.join(zarr_path, "0"), mode="r")
        return np.asarray(z[t])

@dataclass
class TrackGraph:
    node_t: np.ndarray
    node_z: np.ndarray
    node_y: np.ndarray
    node_x: np.ndarray
    node_ids: np.ndarray
    edges: np.ndarray
    meta: dict
    
    @property
    def n_nodes(self) -> int:
        return len(self.node_ids)
    
    @property
    def n_edges(self) -> int:
        return len(self.edges)
    
    def coords_by_id(self) -> dict:
        out = {}
        for i, nid in enumerate(self.node_ids):
            out[int(nid)] = (int(self.node_t[i]), float(self.node_z[i]),
                           float(self.node_y[i]), float(self.node_x[i]))
        return out

def _ball_footprint(radius_um: float, eff_spacing: np.ndarray) -> np.ndarray:
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1,
                          -rad_vox[1]:rad_vox[1]+1,
                          -rad_vox[2]:rad_vox[2]+1]
    d = ((zz * eff_spacing[0])**2 + (yy * eff_spacing[1])**2 + (xx * eff_spacing[2])**2)
    return d <= radius_um**2

print("Data I/O loaded")

Data I/O loaded


In [3]:
def detect_blobs_enhanced(vol: np.ndarray,
                         xy_downsample: int = 2,
                         min_distance_um: float = 2.5,
                         rel_threshold: float = 0.025,
                         abs_percentile: float = 40.0,
                         max_peaks: int = 60000) -> np.ndarray:
    """Enhanced blob detector with multi-scale detection."""
    
    vf = vol.astype(np.float32)
    ds = vf[:, ::xy_downsample, ::xy_downsample]
    
    eff = np.array([SCALE[0], SCALE[1]*xy_downsample, SCALE[2]*xy_downsample])
    
    # Normalize
    lo, hi = np.percentile(ds, [1.0, 99.5])
    if hi <= lo:
        hi = lo + 1.0
    norm = np.clip((ds - lo) / (hi - lo), 0, None)
    
    # Multi-scale detection
    scales = [[1.2, 3.5], [1.8, 5.0], [2.5, 6.5]]
    all_coords = []
    all_scores = []
    
    for small_um, large_um in scales:
        s_small = small_um / eff
        s_large = large_um / eff
        
        g1 = gaussian_filter(norm, sigma=s_small)
        g2 = gaussian_filter(norm, sigma=s_large)
        dog = g1 - g2
        
        footprint = _ball_footprint(min_distance_um, eff)
        mx = maximum_filter(dog, footprint=footprint, mode="nearest")
        
        thr = max(rel_threshold, np.percentile(dog[dog>0], 50) if np.any(dog>0) else 0)
        abs_thr = np.percentile(norm, abs_percentile)
        
        peaks = (dog == mx) & (dog >= thr) & (norm >= abs_thr)
        coords = np.argwhere(peaks)
        
        if len(coords) > 0:
            vals = dog[peaks]
            if len(coords) > max_peaks // len(scales):
                idx = np.argsort(vals)[-max_peaks // len(scales):]
                coords = coords[idx]
                vals = vals[idx]
            
            coords = coords.astype(np.float64)
            coords[:, 1] *= xy_downsample
            coords[:, 2] *= xy_downsample
            
            all_coords.append(coords)
            all_scores.append(vals)
    
    if not all_coords:
        return np.zeros((0, 3), dtype=np.float64)
    
    # Merge peaks
    all_coords = np.vstack(all_coords)
    all_scores = np.concatenate(all_scores)
    
    # Sort by score
    idx = np.argsort(all_scores)[::-1]
    all_coords = all_coords[idx]
    all_scores = all_scores[idx]
    
    # Non-maximum suppression
    keep = []
    for i, coord in enumerate(all_coords):
        if len(keep) == 0:
            keep.append(i)
        else:
            distances = np.sqrt(((all_coords[keep] - coord) * SCALE)**2)
            distances = np.sqrt((distances**2).sum(axis=1))
            if np.min(distances) >= min_distance_um:
                keep.append(i)
        if len(keep) >= max_peaks:
            break
    
    return all_coords[keep]

def refine_centroids(vol: np.ndarray, coords: np.ndarray, win=(1, 3, 3)) -> np.ndarray:
    """Intensity-weighted center of mass refinement."""
    if len(coords) == 0:
        return coords
    
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-wz), min(Z, z+wz+1)
        y0, y1 = max(0, y-wy), min(Y, y+wy+1)
        x0, x1 = max(0, x-wx), min(X, x+wx+1)
        
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0:
            continue
        
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        
        out[i, 0] = (patch * zz).sum() / s
        out[i, 1] = (patch * yy).sum() / s
        out[i, 2] = (patch * xx).sum() / s
    
    return out

print("Detection module loaded")

Detection module loaded


In [4]:
# ============ WATERSHED-BASED DETECTION (alternative to detect_blobs_enhanced) ============
# Same multi-scale DoG peak-finding as before, but now used as MARKERS for a watershed
# segmentation instead of being taken directly as cell centers. This separates touching/dense
# cells much better than pure peak-picking + NMS. Also fixes depth-dependent signal falloff by
# normalizing intensity per z-block instead of globally across the whole volume.
# Returns (coords, scores): coords is (N,3) [z,y,x] in full-resolution voxel units,
# scores is (N,) mean intensity per segment -- carried forward for appearance-aware linking.

from skimage.segmentation import watershed
from skimage.measure import regionprops
from typing import Tuple

def detect_blobs_watershed(vol: np.ndarray,
                            xy_downsample: int = 2,
                            min_distance_um: float = 2.5,
                            rel_threshold: float = 0.025,
                            abs_percentile: float = 40.0,
                            max_peaks: int = 60000,
                            z_block: int = 8,
                            mask_percentile: float = 20.0) -> Tuple[np.ndarray, np.ndarray]:
    """Marker-controlled watershed detector with per-z-block normalization."""
    vf = vol.astype(np.float32)
    ds = vf[:, ::xy_downsample, ::xy_downsample]
    eff = np.array([SCALE[0], SCALE[1] * xy_downsample, SCALE[2] * xy_downsample])

    # --- per-z-block normalization (fixes depth-dependent signal attenuation) ---
    Z = ds.shape[0]
    norm = np.zeros_like(ds, dtype=np.float32)
    for z0 in range(0, Z, z_block):
        z1 = min(Z, z0 + z_block)
        block = ds[z0:z1]
        lo, hi = np.percentile(block, [1.0, 99.5])
        if hi <= lo:
            hi = lo + 1.0
        norm[z0:z1] = np.clip((block - lo) / (hi - lo), 0, None)

    # --- multi-scale DoG peaks, reused here as watershed markers ---
    scales = [[1.2, 3.5], [1.8, 5.0], [2.5, 6.5]]
    all_coords, all_scores, dog_stack = [], [], []
    for small_um, large_um in scales:
        s_small = small_um / eff
        s_large = large_um / eff
        g1 = gaussian_filter(norm, sigma=s_small)
        g2 = gaussian_filter(norm, sigma=s_large)
        dog = g1 - g2
        dog_stack.append(dog)
        footprint = _ball_footprint(min_distance_um, eff)
        mx = maximum_filter(dog, footprint=footprint, mode="nearest")
        thr = max(rel_threshold, np.percentile(dog[dog > 0], 50) if np.any(dog > 0) else 0)
        abs_thr = np.percentile(norm, abs_percentile)
        peaks = (dog == mx) & (dog >= thr) & (norm >= abs_thr)
        coords = np.argwhere(peaks)
        if len(coords):
            vals = dog[peaks]
            all_coords.append(coords.astype(np.float64))
            all_scores.append(vals)

    if not all_coords:
        return np.zeros((0, 3)), np.zeros((0,))

    all_coords = np.vstack(all_coords)
    all_scores = np.concatenate(all_scores)
    idx = np.argsort(all_scores)[::-1]
    all_coords, all_scores = all_coords[idx], all_scores[idx]

    # NMS to pick marker seeds (same policy as detect_blobs_enhanced)
    keep = []
    for i, coord in enumerate(all_coords):
        if not keep:
            keep.append(i)
        else:
            d = np.sqrt((((all_coords[keep] - coord) * eff) ** 2).sum(axis=1))
            if d.min() >= min_distance_um:
                keep.append(i)
        if len(keep) >= max_peaks:
            break
    markers_ds = all_coords[keep].astype(np.int64)
    if len(markers_ds) == 0:
        return np.zeros((0, 3)), np.zeros((0,))

    combined_dog = np.max(np.stack(dog_stack, axis=0), axis=0)
    pos = combined_dog[combined_dog > 0]
    mask_thr = np.percentile(pos, mask_percentile) if pos.size else 0
    mask = combined_dog > mask_thr

    marker_img = np.zeros(ds.shape, dtype=np.int32)
    for i, (z, y, x) in enumerate(markers_ds):
        marker_img[z, y, x] = i + 1
    mask = mask | (marker_img > 0)  # markers always seed a region even if just below mask_thr

    labels = watershed(-combined_dog, markers=marker_img, mask=mask)

    props = regionprops(labels, intensity_image=norm)
    out_coords, out_scores = [], []
    for p in props:
        if p.area < 2:
            continue
        try:
            cz, cy, cx = p.centroid_weighted       # skimage >= 0.24
        except AttributeError:
            cz, cy, cx = p.weighted_centroid        # older skimage (Kaggle images vary)
        try:
            score = float(p.intensity_mean)
        except AttributeError:
            score = float(p.mean_intensity)
        out_coords.append((cz, cy * xy_downsample, cx * xy_downsample))
        out_scores.append(score)

    if not out_coords:
        return np.zeros((0, 3)), np.zeros((0,))
    return np.array(out_coords, dtype=np.float64), np.array(out_scores, dtype=np.float64)

print("Watershed detection module loaded")


Watershed detection module loaded


In [5]:
"""
Division-aware linking module for the Biohub cell tracking pipeline.

Drop-in replacement for `link_motion_enhanced`. Adds a second pass after the
standard 1-to-1 Hungarian assignment that looks for mitosis events: an
existing track whose predicted position is close to TWO next-frame
detections (instead of one) is treated as a division, and BOTH daughters
get an edge from the parent's last node. This is the piece your current
pipeline is missing entirely (it can only ever produce 1-to-1 edges).

Usage: replace the call to `link_motion_enhanced(...)` in `process_dataset`
with `link_motion_with_divisions(...)`. Same signature, extra kwargs at the
end with sane defaults.
"""

import numpy as np
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)  # z, y, x um/voxel


class Track:
    __slots__ = ['pos', 'vel', 'node_id', 'miss', 'alive']
    def __init__(self, pos, node_id):
        self.pos = pos.copy()
        self.vel = np.zeros(3)
        self.node_id = node_id
        self.miss = 0
        self.alive = True


def link_motion_with_divisions(
    frames: list,
    max_link_um: float = 8.0,
    motion_weight: float = 0.7,
    max_miss: int = 2,
    # --- new division-related params ---
    division_radius_um: float = 6.0,     # how close a 2nd daughter candidate must be to the parent's predicted pos
    division_min_gap_um: float = 1.0,    # daughters must be at least this far apart from each other (else it's just noise/jitter)
    division_symmetry_tol: float = 0.6,  # max allowed relative difference in distance-from-parent between the two daughters (0=perfectly symmetric)
    max_daughters_per_parent: int = 2,
):
    """
    Frame-to-frame linking with basic mitosis detection.

    Strategy per transition t -> t+1:
      1. Standard Hungarian assignment on scaled centroid distance
         (same as before) gives each track its single best-match daughter.
      2. For every UNMATCHED detection in frame t+1, check every ALIVE
         track's predicted position. If the unmatched detection is within
         `division_radius_um` of a track that already got a match this
         frame, AND the two candidate daughters are roughly symmetric in
         distance from the parent (a coarse but decent proxy for "these
         two cells emerged from the same parent" vs "this is just a
         different nearby cell"), record a second edge from the parent's
         last node -> this detection. The track continues as BOTH children
         (division = branching, not termination), so we spawn a new Track
         object for the second daughter.
    """
    node_ids, node_t, node_z, node_y, node_x = [], [], [], [], []
    nid = 1
    frame_ids = []
    tracks = []

    for t, coords in enumerate(frames):
        ids_t = []
        for coord in coords:
            node_ids.append(nid)
            node_t.append(t)
            node_z.append(coord[0]); node_y.append(coord[1]); node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0:
                tracks.append(Track(np.asarray(coord, dtype=np.float64), nid))
            nid += 1
        frame_ids.append(ids_t)

    def empty_graph():
        return dict(
            node_t=np.array(node_t, dtype=np.int64),
            node_z=np.array(node_z, dtype=np.float64),
            node_y=np.array(node_y, dtype=np.float64),
            node_x=np.array(node_x, dtype=np.float64),
            node_ids=np.array(node_ids, dtype=np.int64),
            edges=np.array([], dtype=np.int64).reshape(-1, 2),
        )

    if len(frames) <= 1:
        return empty_graph()

    edges = []

    for t in range(1, len(frames)):
        current_coords = frames[t]
        current_ids = frame_ids[t]

        alive = [tr for tr in tracks if tr.alive]

        if len(current_coords) == 0 or len(alive) == 0:
            for tr in alive:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False
            continue

        cur = np.asarray(current_coords, dtype=np.float64)
        predicted = np.array([tr.pos + tr.vel for tr in alive])

        d_scaled = ((predicted[:, None, :] - cur[None, :, :]) * SCALE).astype(np.float64)
        dist = np.sqrt((d_scaled ** 2).sum(axis=2))  # (n_tracks, n_detections)

        cost = np.where(dist <= max_link_um, dist, 1e6)
        row_ind, col_ind = linear_sum_assignment(cost)

        matched_track_for_det = {}   # det_idx -> track_idx (primary match)
        used_dets = set()

        for r, c in zip(row_ind, col_ind):
            if dist[r, c] > max_link_um:
                continue
            tr = alive[r]
            new_pos = cur[c]
            edges.append((tr.node_id, current_ids[c]))
            tr.vel = motion_weight * (new_pos - tr.pos) + (1 - motion_weight) * tr.vel
            tr.pos = new_pos
            tr.node_id = current_ids[c]
            tr.miss = 0
            matched_track_for_det[c] = r
            used_dets.add(c)

        # unmatched tracks: predict forward, allow a few missed frames
        matched_track_rows = set(matched_track_for_det.values())
        for r, tr in enumerate(alive):
            if r not in matched_track_rows:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False

        # --- Division pass: look for a 2nd daughter for tracks that already matched ---
        unmatched_dets = [c for c in range(len(cur)) if c not in used_dets]
        if unmatched_dets and matched_track_for_det:
            new_tracks = []
            daughter_count = defaultdict(lambda: 1)  # primary match already counts as 1

            for c in unmatched_dets:
                best_r, best_dist = None, None
                for c2, r in matched_track_for_det.items():
                    if daughter_count[r] >= max_daughters_per_parent:
                        continue
                    tr = alive[r]
                    d_parent = np.sqrt((((tr.pos - cur[c]) * SCALE) ** 2).sum())
                    if d_parent > division_radius_um:
                        continue
                    # symmetry check vs the primary daughter already assigned to this parent
                    d_primary = dist[r, c2]
                    if d_primary <= 1e-9:
                        continue
                    asym = abs(d_parent - d_primary) / max(d_parent, d_primary)
                    d_between = np.sqrt(((( cur[c2] - cur[c]) * SCALE) ** 2).sum())
                    if d_between < division_min_gap_um:
                        continue
                    if asym > division_symmetry_tol:
                        continue
                    if best_dist is None or d_parent < best_dist:
                        best_r, best_dist = r, d_parent

                if best_r is not None:
                    parent_tr = alive[best_r]
                    edges.append((parent_tr.node_id, current_ids[c]))
                    daughter_count[best_r] += 1
                    # spawn a new track for this second daughter
                    new_tr = Track(cur[c], current_ids[c])
                    new_tr.vel = parent_tr.vel.copy()
                    new_tracks.append(new_tr)
                    used_dets.add(c)

            tracks.extend(new_tracks)

        # any still-unmatched detections become brand new tracks (new cell entering FOV, etc.)
        for c in range(len(cur)):
            if c not in used_dets:
                tracks.append(Track(cur[c], current_ids[c]))

        tracks = [tr for tr in tracks if tr.alive] + [tr for tr in tracks if not tr.alive and tr not in tracks]
        # (keep dead tracks out; simpler: just filter alive, dead ones drop naturally next loop since we rebuild `alive` from `tracks`)
        tracks = [tr for tr in tracks if tr.alive]

    g = empty_graph()
    g["edges"] = np.array(edges, dtype=np.int64).reshape(-1, 2) if edges else np.array([], dtype=np.int64).reshape(-1, 2)
    return g

In [6]:
# ============ APPEARANCE-AWARE, DIVISION-PRESERVING LINKER ============
# Drop-in upgrade to link_motion_with_divisions. Same division-detection logic (unchanged),
# but the primary Hungarian assignment cost now also penalizes mismatched intensity/score
# between a track's predicted appearance and each candidate detection -- this disambiguates
# two similarly-positioned cells in dense regions where centroid distance alone is ambiguous.
# Needs per-frame scores (frame_scores[t][i] lines up with frames[t][i]); if you're not using
# detect_blobs_watershed, pass frame_scores = [np.ones(len(f)) for f in frames] to disable the
# appearance term (appearance_weight has no effect when all scores are equal).

class TrackV2:
    __slots__ = ['pos', 'vel', 'score', 'node_id', 'miss', 'alive']
    def __init__(self, pos, score, node_id):
        self.pos = pos.copy()
        self.vel = np.zeros(3)
        self.score = score
        self.node_id = node_id
        self.miss = 0
        self.alive = True


def link_motion_with_divisions_v2(
    frames: list,
    frame_scores: list,
    max_link_um: float = 8.0,
    motion_weight: float = 0.7,
    max_miss: int = 2,
    appearance_weight: float = 4.0,
    division_radius_um: float = 6.0,
    division_min_gap_um: float = 1.0,
    division_symmetry_tol: float = 0.6,
    max_daughters_per_parent: int = 2,
):
    node_ids, node_t, node_z, node_y, node_x = [], [], [], [], []
    nid = 1
    frame_ids = []
    tracks = []

    for t, coords in enumerate(frames):
        ids_t = []
        scores_t = frame_scores[t]
        for i, coord in enumerate(coords):
            node_ids.append(nid)
            node_t.append(t)
            node_z.append(coord[0]); node_y.append(coord[1]); node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0:
                tracks.append(TrackV2(np.asarray(coord, dtype=np.float64), float(scores_t[i]), nid))
            nid += 1
        frame_ids.append(ids_t)

    def empty_graph():
        return dict(
            node_t=np.array(node_t, dtype=np.int64),
            node_z=np.array(node_z, dtype=np.float64),
            node_y=np.array(node_y, dtype=np.float64),
            node_x=np.array(node_x, dtype=np.float64),
            node_ids=np.array(node_ids, dtype=np.int64),
            edges=np.array([], dtype=np.int64).reshape(-1, 2),
        )

    if len(frames) <= 1:
        return empty_graph()

    # normalize the score scale once so appearance_weight is comparable across datasets
    nonempty = [np.asarray(s) for s in frame_scores if len(s)]
    all_scores_flat = np.concatenate(nonempty) if nonempty else np.array([1.0])
    score_scale = np.std(all_scores_flat) + 1e-6

    edges = []

    for t in range(1, len(frames)):
        current_coords = frames[t]
        current_scores = frame_scores[t]
        current_ids = frame_ids[t]

        alive = [tr for tr in tracks if tr.alive]

        if len(current_coords) == 0 or len(alive) == 0:
            for tr in alive:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False
            continue

        cur = np.asarray(current_coords, dtype=np.float64)
        cur_scores = np.asarray(current_scores, dtype=np.float64)
        predicted = np.array([tr.pos + tr.vel for tr in alive])
        pred_scores = np.array([tr.score for tr in alive])

        d_scaled = ((predicted[:, None, :] - cur[None, :, :]) * SCALE).astype(np.float64)
        dist = np.sqrt((d_scaled ** 2).sum(axis=2))
        appearance_diff = np.abs(pred_scores[:, None] - cur_scores[None, :]) / score_scale

        cost = np.where(dist <= max_link_um, dist + appearance_weight * appearance_diff, 1e6)
        row_ind, col_ind = linear_sum_assignment(cost)

        matched_track_for_det = {}
        used_dets = set()

        for r, c in zip(row_ind, col_ind):
            if dist[r, c] > max_link_um:
                continue
            tr = alive[r]
            new_pos = cur[c]
            edges.append((tr.node_id, current_ids[c]))
            tr.vel = motion_weight * (new_pos - tr.pos) + (1 - motion_weight) * tr.vel
            tr.pos = new_pos
            tr.score = 0.7 * tr.score + 0.3 * cur_scores[c]
            tr.node_id = current_ids[c]
            tr.miss = 0
            matched_track_for_det[c] = r
            used_dets.add(c)

        matched_track_rows = set(matched_track_for_det.values())
        for r, tr in enumerate(alive):
            if r not in matched_track_rows:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False

        # --- division pass: unchanged from link_motion_with_divisions ---
        unmatched_dets = [c for c in range(len(cur)) if c not in used_dets]
        if unmatched_dets and matched_track_for_det:
            new_tracks = []
            daughter_count = defaultdict(lambda: 1)

            for c in unmatched_dets:
                best_r, best_dist = None, None
                for c2, r in matched_track_for_det.items():
                    if daughter_count[r] >= max_daughters_per_parent:
                        continue
                    tr = alive[r]
                    d_parent = np.sqrt((((tr.pos - cur[c]) * SCALE) ** 2).sum())
                    if d_parent > division_radius_um:
                        continue
                    d_primary = dist[r, c2]
                    if d_primary <= 1e-9:
                        continue
                    asym = abs(d_parent - d_primary) / max(d_parent, d_primary)
                    d_between = np.sqrt(((( cur[c2] - cur[c]) * SCALE) ** 2).sum())
                    if d_between < division_min_gap_um:
                        continue
                    if asym > division_symmetry_tol:
                        continue
                    if best_dist is None or d_parent < best_dist:
                        best_r, best_dist = r, d_parent

                if best_r is not None:
                    parent_tr = alive[best_r]
                    edges.append((parent_tr.node_id, current_ids[c]))
                    daughter_count[best_r] += 1
                    new_tr = TrackV2(cur[c], cur_scores[c], current_ids[c])
                    new_tr.vel = parent_tr.vel.copy()
                    new_tracks.append(new_tr)
                    used_dets.add(c)

            tracks.extend(new_tracks)

        for c in range(len(cur)):
            if c not in used_dets:
                tracks.append(TrackV2(cur[c], cur_scores[c], current_ids[c]))

        tracks = [tr for tr in tracks if tr.alive]

    g = empty_graph()
    g["edges"] = np.array(edges, dtype=np.int64).reshape(-1, 2) if edges else np.array([], dtype=np.int64).reshape(-1, 2)
    return g

print("Appearance-aware linking module loaded")


Appearance-aware linking module loaded


In [7]:
def link_motion_enhanced(frames: list, 
                        max_link_um: float = 8.0,
                        motion_weight: float = 0.7,
                        max_miss: int = 2) -> TrackGraph:
    """Enhanced motion-based tracking with velocity prediction."""
    
    node_ids = []
    node_t = []
    node_z = []
    node_y = []
    node_x = []
    frame_ids = []
    nid = 1
    
    # Initialize tracks
    class Track:
        __slots__ = ['pos', 'vel', 'node_id', 'miss', 'positions', 'alive']
        def __init__(self, pos, node_id):
            self.pos = pos.copy()
            self.vel = np.zeros(3)
            self.node_id = node_id
            self.miss = 0
            self.positions = [pos.copy()]
            self.alive = True
    
    # First frame
    tracks = []
    for t, coords in enumerate(frames):
        ids_t = []
        for coord in coords:
            node_ids.append(nid)
            node_t.append(t)
            node_z.append(coord[0])
            node_y.append(coord[1])
            node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0:
                tracks.append(Track(coord, nid))
            nid += 1
        frame_ids.append(ids_t)
    
    if len(frames) == 1:
        return TrackGraph(
            node_t=np.array(node_t, dtype=np.int64),
            node_z=np.array(node_z, dtype=np.float64),
            node_y=np.array(node_y, dtype=np.float64),
            node_x=np.array(node_x, dtype=np.float64),
            node_ids=np.array(node_ids, dtype=np.int64),
            edges=np.array([], dtype=np.int64).reshape(-1, 2),
            meta={}
        )
    
    edges = []
    
    for t in range(1, len(frames)):
        current_coords = frames[t]
        if len(current_coords) == 0:
            # No detections, just predict forward
            for track in tracks:
                track.miss += 1
                track.pos = track.pos + track.vel
                if track.miss > max_miss:
                    track.alive = False
            tracks = [t for t in tracks if t.alive]
            continue
        
        # Predict positions
        pred_pos = []
        for track in tracks:
            if len(track.positions) >= 2:
                vel = track.positions[-1] - track.positions[-2]
                track.vel = 0.6 * vel + 0.4 * track.vel
            pred = track.pos + track.vel * (1 + track.miss * 0.1)
            pred_pos.append(pred)
        
        # Match
        if tracks:
            cost_matrix = np.zeros((len(tracks), len(current_coords)))
            for i, pred in enumerate(pred_pos):
                max_dist = max_link_um * (1 + tracks[i].miss * 0.2)
                for j, coord in enumerate(current_coords):
                    dist = np.linalg.norm((pred - coord) * SCALE)
                    cost_matrix[i, j] = dist if dist <= max_dist else 1e6
            
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            matched_tracks = set()
            matched_dets = set()
            
            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < 1e6:
                    edges.append((tracks[r].node_id, frame_ids[t][c]))
                    tracks[r].pos = current_coords[c].copy()
                    tracks[r].positions.append(current_coords[c].copy())
                    tracks[r].node_id = frame_ids[t][c]
                    tracks[r].miss = 0
                    matched_tracks.add(r)
                    matched_dets.add(c)
        
        # Update unmatched tracks
        for i, track in enumerate(tracks):
            if i not in matched_tracks:
                track.miss += 1
                track.pos = track.pos + track.vel
                if track.miss > max_miss:
                    track.alive = False
        
        # Remove dead tracks
        tracks = [t for t in tracks if t.alive]
        
        # Add new tracks for unmatched detections
        for c in range(len(current_coords)):
            if c not in matched_dets:
                new_track = Track(current_coords[c], frame_ids[t][c])
                tracks.append(new_track)
    
    # Build graph
    return TrackGraph(
        node_t=np.array(node_t, dtype=np.int64),
        node_z=np.array(node_z, dtype=np.float64),
        node_y=np.array(node_y, dtype=np.float64),
        node_x=np.array(node_x, dtype=np.float64),
        node_ids=np.array(node_ids, dtype=np.int64),
        edges=np.array(edges, dtype=np.int64).reshape(-1, 2),
        meta={}
    )

print("Tracking module loaded")

Tracking module loaded


In [8]:
from scipy.ndimage import gaussian_filter1d

def close_gaps_enhanced(frames: list, g: TrackGraph, max_gap: int = 2,
                       gap_dist_um: float = 8.0) -> TrackGraph:
    """Enhanced gap closing with interpolation."""
    if g.n_edges == 0:
        return g
    
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    
    ends_by_t = defaultdict(list)
    starts_by_t = defaultdict(list)
    
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out:
            ends_by_t[t].append(nid)
        if nid not in has_in:
            starts_by_t[t].append(nid)
    
    new_nodes = []
    new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1, [])
            if not starts:
                continue
            
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            
            if len(ec) == 0 or len(sc) == 0:
                continue
            
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :])**2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            cost = np.where(d <= thr, d, 1e6)
            
            row_ind, col_ind = linear_sum_assignment(cost)
            used_s = set()
            
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s:
                    continue
                
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]
                ts, zs, ys, xs = coords[s_id]
                
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac
                    yi = ye + (ys - ye) * frac
                    xi = xe + (xs - xe) * frac
                    nid = next_id
                    next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid))
                    new_edges.append((prev, nid))
                    prev = nid
                new_edges.append((prev, s_id))
                has_out.add(e_id)
                used_s.add(c)
    
    if not new_nodes:
        return g
    
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, 
                     node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g: TrackGraph) -> TrackGraph:
    """Remove nodes not referenced by any edge."""
    if g.n_edges == 0:
        return g
    
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    
    if len(keep) == len(g.node_ids):
        return g
    
    return TrackGraph(
        node_t=g.node_t[keep], node_z=g.node_z[keep], 
        node_y=g.node_y[keep], node_x=g.node_x[keep],
        node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta
    )

def smooth_tracks(g: TrackGraph) -> TrackGraph:
    """Smooth track positions using temporal filtering."""
    if g.n_nodes < 5:
        return g
    
    # Build adjacency
    adj = defaultdict(list)
    for src, tgt in g.edges:
        adj[src].append(tgt)
        adj[tgt].append(src)
    
    # Find tracks (connected components)
    visited = set()
    tracks = []
    
    for node in g.node_ids:
        if node in visited:
            continue
        stack = [node]
        track = []
        while stack:
            curr = stack.pop()
            if curr in visited:
                continue
            visited.add(curr)
            track.append(curr)
            for neighbor in adj[curr]:
                if neighbor not in visited:
                    stack.append(neighbor)
        if len(track) >= 5:
            tracks.append(track)
    
    # Smooth each track
    for track in tracks:
        # Get positions and times
        positions = []
        times = []
        for nid in track:
            idx = np.where(g.node_ids == nid)[0][0]
            positions.append([g.node_z[idx], g.node_y[idx], g.node_x[idx]])
            times.append(g.node_t[idx])
        
        positions = np.array(positions)
        
        # Check if times are sequential
        if len(np.unique(times)) == len(times):
            # Smooth positions
            smoothed = gaussian_filter1d(positions, sigma=0.5, axis=0, mode='nearest')
            
            # Update positions if not too far
            for i, nid in enumerate(track):
                idx = np.where(g.node_ids == nid)[0][0]
                dist = np.linalg.norm((positions[i] - smoothed[i]) * SCALE)
                if dist < 3.0:
                    g.node_z[idx] = smoothed[i, 0]
                    g.node_y[idx] = smoothed[i, 1]
                    g.node_x[idx] = smoothed[i, 2]
    
    return g

print("Post-processing module loaded")

Post-processing module loaded


In [9]:
def detect_divisions_simple(g: TrackGraph) -> TrackGraph:
    """
    Simple division detection: find nodes with exactly 2 outgoing edges
    where children appear in consecutive frames.
    """
    if g.n_edges == 0:
        return g
    
    out_edges = {}
    for src, tgt in g.edges:
        out_edges.setdefault(src, []).append(tgt)
    
    divisions = []
    for node_id in g.node_ids:
        children = out_edges.get(node_id, [])
        if len(children) == 2:
            # Check if children appear in next frame
            idx = np.where(g.node_ids == node_id)[0][0]
            parent_t = g.node_t[idx]
            
            child_times = []
            for child in children:
                child_idx = np.where(g.node_ids == child)[0]
                if len(child_idx) > 0:
                    child_times.append(g.node_t[child_idx[0]])
            
            if len(child_times) == 2 and all(t == parent_t + 1 for t in child_times):
                divisions.append({
                    'node_id': int(node_id),
                    'children': [int(c) for c in children],
                    't': int(parent_t)
                })
    
    g.meta['divisions'] = divisions
    print(f"  ✅ Detected {len(divisions)} simple divisions")
    return g

In [10]:
def analyze_divisions(g: TrackGraph) -> List[int]:
    """
    Analyze division nodes in the graph and print statistics.
    Returns list of division node IDs.
    """
    if g.n_edges == 0:
        print("  No edges in graph")
        return []

    out_edges = {}
    for src, tgt in g.edges:
        out_edges.setdefault(src, []).append(tgt)

    div_nodes = [nid for nid in g.node_ids if len(out_edges.get(nid, [])) == 2]

    if len(div_nodes) > 0:
        print(f"  \U0001f52c Total division nodes: {len(div_nodes)}")
        print("  Sample division nodes:")
        for nid in div_nodes[:5]:
            idx = np.where(g.node_ids == nid)[0][0]
            children = out_edges[nid]
            parent_t = g.node_t[idx]
            child_times = []
            for child in children:
                child_idx = np.where(g.node_ids == child)[0]
                if len(child_idx) > 0:
                    child_times.append(g.node_t[child_idx[0]])
            print(f"    Node {nid}: t={parent_t}, children={children}, child_times={child_times}")
    else:
        print("  \u2139\ufe0f No division nodes found")

    return div_nodes

def process_dataset(zarr_path: str, **kwargs) -> TrackGraph:
    """
    Process a dataset with flexible parameters.
    All parameters are passed via kwargs with defaults.

    New flags added to enrich the pipeline (see CONFIG cell):
      use_watershed       -- detect via marker-controlled watershed instead of raw peak-picking
      use_appearance_cost -- add an intensity-similarity term to the linking cost
      z_block             -- z-block size (voxels) for per-depth intensity normalization
      mask_percentile     -- watershed foreground mask threshold (percentile of positive DoG)
      appearance_weight   -- how strongly intensity mismatch penalizes a candidate link
    """
    # Extract parameters with defaults
    xy_downsample = kwargs.get('xy_downsample', 2)
    min_distance_um = kwargs.get('min_distance_um', 2.5)
    rel_threshold = kwargs.get('rel_threshold', 0.025)
    abs_percentile = kwargs.get('abs_percentile', 40.0)
    max_peaks = kwargs.get('max_peaks', 60000)
    max_link_um = kwargs.get('max_link_um', 8.0)
    motion_weight = kwargs.get('motion_weight', 0.7)
    max_miss = kwargs.get('max_miss', 2)
    close_gaps = kwargs.get('close_gaps', True)
    max_gap = kwargs.get('max_gap', 2)
    gap_dist_um = kwargs.get('gap_dist_um', 8.0)
    refine = kwargs.get('refine', True)
    smooth = kwargs.get('smooth', True)

    # Division parameters
    division_radius_um = kwargs.get('division_radius_um', 8.0)
    division_min_gap_um = kwargs.get('division_min_gap_um', 0.5)
    division_symmetry_tol = kwargs.get('division_symmetry_tol', 0.8)
    max_daughters_per_parent = kwargs.get('max_daughters_per_parent', 2)

    # --- new flags ---
    use_watershed = kwargs.get('use_watershed', False)
    use_appearance_cost = kwargs.get('use_appearance_cost', False)
    z_block = kwargs.get('z_block', 8)
    mask_percentile = kwargs.get('mask_percentile', 20.0)
    appearance_weight = kwargs.get('appearance_weight', 4.0)

    # Load volume
    vol_meta = open_image(zarr_path)
    n_t = vol_meta.n_t

    # Detect cells in each frame
    frames = []
    frames_scores = []
    for t in range(n_t):
        vol = vol_meta.frame(t)

        if use_watershed:
            coords, scores = detect_blobs_watershed(
                vol,
                xy_downsample=xy_downsample,
                min_distance_um=min_distance_um,
                rel_threshold=rel_threshold,
                abs_percentile=abs_percentile,
                max_peaks=max_peaks,
                z_block=z_block,
                mask_percentile=mask_percentile,
            )
            # the watershed centroid is already intensity-weighted within its segment;
            # skip the extra fixed-window refine pass unless explicitly requested
            if refine and kwargs.get('refine_after_watershed', False) and len(coords) > 0:
                coords = refine_centroids(vol, coords)
        else:
            coords = detect_blobs_enhanced(
                vol,
                xy_downsample=xy_downsample,
                min_distance_um=min_distance_um,
                rel_threshold=rel_threshold,
                abs_percentile=abs_percentile,
                max_peaks=max_peaks
            )
            if refine and len(coords) > 0:
                coords = refine_centroids(vol, coords)
            scores = np.ones(len(coords), dtype=np.float64)

        frames.append(coords)
        frames_scores.append(scores)

        # Free memory
        del vol
        if t % 10 == 0:
            gc.collect()

    # Link frames WITH DIVISIONS (appearance-aware cost is optional)
    if use_appearance_cost:
        g_dict = link_motion_with_divisions_v2(
            frames,
            frames_scores,
            max_link_um=max_link_um,
            motion_weight=motion_weight,
            max_miss=max_miss,
            appearance_weight=appearance_weight,
            division_radius_um=division_radius_um,
            division_min_gap_um=division_min_gap_um,
            division_symmetry_tol=division_symmetry_tol,
            max_daughters_per_parent=max_daughters_per_parent
        )
    else:
        g_dict = link_motion_with_divisions(
            frames,
            max_link_um=max_link_um,
            motion_weight=motion_weight,
            max_miss=max_miss,
            division_radius_um=division_radius_um,
            division_min_gap_um=division_min_gap_um,
            division_symmetry_tol=division_symmetry_tol,
            max_daughters_per_parent=max_daughters_per_parent
        )
    g = TrackGraph(**g_dict, meta={})

    # Analyze divisions BEFORE post-processing
    print("  Analyzing divisions...")
    div_nodes = analyze_divisions(g)
    if len(div_nodes) > 0:
        g.meta['divisions'] = [{'node_id': int(nid)} for nid in div_nodes]

    # Post-process
    if close_gaps:
        g = close_gaps_enhanced(frames, g, max_gap=max_gap, gap_dist_um=gap_dist_um)

    g = prune_isolated(g)

    if smooth:
        g = smooth_tracks(g)

    # Analyze divisions AFTER post-processing
    print("  Analyzing divisions after post-processing...")
    div_nodes_after = analyze_divisions(g)
    if len(div_nodes_after) > 0:
        g.meta['divisions'] = [{'node_id': int(nid)} for nid in div_nodes_after]

    return g

print("\u2705 Updated process_dataset with watershed detection + appearance-aware linking (toggle via CONFIG)")


✅ Updated process_dataset with watershed detection + appearance-aware linking (toggle via CONFIG)


In [11]:
def add_division_metadata(g: TrackGraph) -> TrackGraph:
    """
    Add division metadata to the graph for the scorer.
    Finds nodes with exactly 2 outgoing edges and stores them in meta.
    """
    if g.n_edges == 0:
        return g
    
    # Build outgoing edges
    out_edges = {}
    for src, tgt in g.edges:
        if src not in out_edges:
            out_edges[src] = []
        out_edges[src].append(tgt)
    
    divisions = []
    for node_id in g.node_ids:
        children = out_edges.get(node_id, [])
        if len(children) == 2:
            idx = np.where(g.node_ids == node_id)[0][0]
            divisions.append({
                'node_id': int(node_id),
                'children': [int(c) for c in children],
                't': int(g.node_t[idx])
            })
    
    # Store in meta with proper format for scorer
    g.meta['divisions'] = divisions
    
    if len(divisions) > 0:
        print(f"  ✅ Marked {len(divisions)} divisions in metadata")
    
    return g

In [12]:
def graph_to_rows(name: str, g: TrackGraph) -> list:
    """Convert graph to submission rows."""
    rows = []
    
    # Node rows
    for i in range(g.n_nodes):
        rows.append({
            "dataset": name,
            "row_type": "node",
            "node_id": int(g.node_ids[i]),
            "t": int(g.node_t[i]),
            "z": int(round(g.node_z[i])),
            "y": int(round(g.node_y[i])),
            "x": int(round(g.node_x[i])),
            "source_id": -1,
            "target_id": -1,
        })
    
    # Edge rows
    for src, tgt in g.edges:
        rows.append({
            "dataset": name,
            "row_type": "edge",
            "node_id": -1,
            "t": -1,
            "z": -1,
            "y": -1,
            "x": -1,
            "source_id": int(src),
            "target_id": int(tgt),
        })
    
    return rows

def create_submission(results: dict, output_path: str) -> pd.DataFrame:
    """Create submission file."""
    all_rows = []
    for name, g in results.items():
        all_rows.extend(graph_to_rows(name, g))
    
    df = pd.DataFrame(all_rows, columns=[
        "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"
    ])
    df.index.name = "id"
    df.to_csv(output_path)
    return df

print("Submission generator loaded")

Submission generator loaded


In [13]:
# ============================================================
# CONFIGURATION -- tune these against the local validation harness, not by guessing.
# New flags: use_watershed / use_appearance_cost / z_block / mask_percentile / appearance_weight
# Both default to False so this cell reproduces your previous behavior until you flip them on
# and re-run evaluate_on_train(...) to confirm they actually help on YOUR data.
# ============================================================

CONFIG = {
    # Detection
    'xy_downsample': 4,
    'min_distance_um': 4.0,
    'rel_threshold': 0.055,
    'abs_percentile': 50.0,
    'max_peaks': 25000,

    # --- new: watershed detector ---
    'use_watershed': True,        # flip to True to use detect_blobs_watershed instead
    'z_block': 8,                  # z-block size (voxels) for per-depth normalization
    'mask_percentile': 20.0,       # lower = more permissive watershed foreground mask

    # Linking
    'max_link_um': 6.5,
    'motion_weight': 0.85,
    'max_miss': 1,

    # --- new: appearance-aware linking cost ---
    'use_appearance_cost': True,  # flip to True to use link_motion_with_divisions_v2
    'appearance_weight': 4.0,      # higher = intensity mismatch penalized more in linking cost

    # Division
    'division_radius_um': 5.0,
    'division_min_gap_um': 2.0,
    'division_symmetry_tol': 0.4,
    'max_daughters_per_parent': 2,

    # Post-processing
    'close_gaps': True,
    'max_gap': 1,
    'gap_dist_um': 5.0,
    'refine': True,
    'smooth': True,
}

# ============================================================
# FIND TEST DIRECTORY
# ============================================================

def find_test_dir():
    # Check common locations
    candidates = [
        "/kaggle/input/biohub-cell-tracking-during-development/test",
        "/kaggle/input/competitions/biohub-cell-tracking-during-development/test",
        os.environ.get("TEST_DIR", ""),
    ]

    for c in candidates:
        if c and os.path.isdir(c):
            return c

    # Search
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "test":
            zarrs = [d for d in dirs if d.endswith(".zarr")]
            if zarrs:
                return root

    raise FileNotFoundError("Test directory not found")

print("=" * 60)
print("CONFIGURATION READY")
print("=" * 60)
print(f"  Detection threshold: {CONFIG['rel_threshold']}")
print(f"  Max peaks: {CONFIG['max_peaks']}")
print(f"  Min distance: {CONFIG['min_distance_um']} um")
print(f"  Max link distance: {CONFIG['max_link_um']} um")
print(f"  Division radius: {CONFIG['division_radius_um']} um")
print(f"  Division symmetry tolerance: {CONFIG['division_symmetry_tol']}")
print(f"  use_watershed: {CONFIG['use_watershed']}   use_appearance_cost: {CONFIG['use_appearance_cost']}")


CONFIGURATION READY
  Detection threshold: 0.055
  Max peaks: 25000
  Min distance: 4.0 um
  Max link distance: 6.5 um
  Division radius: 5.0 um
  Division symmetry tolerance: 0.4
  use_watershed: True   use_appearance_cost: True


In [14]:
def main():
    print("Starting Biohub Cell Tracking Pipeline...")
    start_time = time.time()
    
    # Find test directory
    test_dir = find_test_dir()
    print(f"Test directory: {test_dir}")
    
    # Get datasets
    datasets = sorted([d[:-5] for d in os.listdir(test_dir) if d.endswith(".zarr")])
    print(f"Found {len(datasets)} datasets")
    
    # Process each dataset
    results = {}
    for i, name in enumerate(datasets):
        print(f"\nProcessing {i+1}/{len(datasets)}: {name}")
        t0 = time.time()
        
        zarr_path = os.path.join(test_dir, name + ".zarr")
        
        try:
            g = process_dataset(zarr_path, **CONFIG)
            results[name] = g
            print(f"  Nodes: {g.n_nodes}, Edges: {g.n_edges}")
            print(f"  Time: {time.time() - t0:.1f}s")
        except Exception as e:
            print(f"  ERROR processing {name}: {e}")
            # Create empty graph for failed dataset
            results[name] = TrackGraph(
                node_t=np.array([], dtype=np.int64),
                node_z=np.array([], dtype=np.float64),
                node_y=np.array([], dtype=np.float64),
                node_x=np.array([], dtype=np.float64),
                node_ids=np.array([], dtype=np.int64),
                edges=np.array([], dtype=np.int64).reshape(-1, 2),
                meta={}
            )
        
        # Free memory
        gc.collect()
    
    # Create submission
    print("\nCreating submission...")
    df = create_submission(results, "submission.csv")
    
    total_time = time.time() - start_time
    print(f"\nDone! Total time: {total_time:.1f}s")
    print(f"Submission rows: {len(df)}")
    print(f"Output file: submission.csv")
    
    # Show sample
    print("\nSample submission:")
    print(df.head(10))

# Run the pipeline
if __name__ == "__main__":
    main()

Starting Biohub Cell Tracking Pipeline...
Test directory: /kaggle/input/competitions/biohub-cell-tracking-during-development/test
Found 4 datasets

Processing 1/4: 44b6_0113de3b
  Analyzing divisions...
  🔬 Total division nodes: 2
  Sample division nodes:
    Node 3482: t=15, children=[np.int64(3479), np.int64(3658)], child_times=[np.int64(15), np.int64(16)]
    Node 20713: t=80, children=[np.int64(20758), np.int64(21056)], child_times=[np.int64(80), np.int64(81)]
  Analyzing divisions after post-processing...
  🔬 Total division nodes: 2
  Sample division nodes:
    Node 3482: t=15, children=[np.int64(3479), np.int64(3658)], child_times=[np.int64(15), np.int64(16)]
    Node 20713: t=80, children=[np.int64(20758), np.int64(21056)], child_times=[np.int64(80), np.int64(81)]
  Nodes: 26136, Edges: 24604
  Time: 52.4s

Processing 2/4: 44b6_0b24845f
  Analyzing divisions...
  🔬 Total division nodes: 51
  Sample division nodes:
    Node 654: t=1, children=[np.int64(762), np.int64(1117)], chil